<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Decision trees

**[Decision trees](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/)** · Machine Learning: From Problem to Reliable Model · Module 5, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** grow a decision tree by hand and with scikit-learn, compare it fairly with logistic regression on the same folds, and add the trees to your experiment log.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The experiment log, from [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log). Cross-validation, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/#cross-validation). |
| **You do not need** | The local project. The setup below downloads the data, writes `ticket_model.py` and writes your experiment log so far. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M05-L02-decision-trees/decision-trees-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: ticket_model.py

The next cell writes `ticket_model.py` as you wrote it in [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/#write-ticket_modelpy): the column lists, `load()` and `build_pipeline()`. Run it. You should see `Writing ticket_model.py` (or `Overwriting`).

In [ ]:
%%writefile ticket_model.py
"""The shared parts of Larkfield's escalation model: the columns, how to load
a part of the data, and the preprocessing-plus-model pipeline.

Nextia Learning, C05, Module 4, lesson 3 (Preprocess with a pipeline).
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")
CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d",
           "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"


def read(path):
    """Read a ticket CSV file. Only an empty cell is missing: the text
    "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


def load(part):
    """One part of the split: train, valid or test."""
    return read(DATA / f"{part}.csv")


def build_pipeline(model=None):
    """Preprocessing and model in one object. fit() learns the medians, the
    scaling and the categories from the rows it is given, and nothing else."""
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", model if model is not None else LogisticRegression(max_iter=1000))])

## Setup: the experiment log

The next cell defines two helpers for your **experiment log**, the file `experiment_log.csv` in `ticket-model/`. You start the log in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log).

- `log_experiment(id, change, cv_ap, cv_std, valid_ap, decision, reason)` adds one row, with today's date. If the id is already in the log, it replaces that row, so you can run a cell again.
- `show_log()` shows the log as a table.

It also writes the 5 rows that you added in the earlier lessons (E01 to E05), so this notebook works without them. If you keep your own log in the local project, skip the last line of the cell.

> **Check.** You should see the log with 5 rows, from E01 to E05.

In [ ]:
# The experiment log: one row per experiment, in experiment_log.csv.
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]


def log_experiment(id, change, cv_ap, cv_std, valid_ap=None, decision="", reason=""):
    """Add one row to the log (or replace the row with the same id)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_ap": round(float(cv_ap), 3), "cv_std": round(float(cv_std), 3),
           "valid_ap": None if valid_ap is None else round(float(valid_ap), 3),
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)


def show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision")):
    """The log as a table (without the date and the reason, unless you ask for them)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}).sort_values("id")
    with pd.option_context("display.max_colwidth", 60, "display.width", 200):
        print(log[list(columns)].fillna("").to_string(index=False))


# The rows of the earlier lessons (id, change, cv_ap, cv_std, valid_ap, decision, reason).
EARLIER_ROWS = [
    ('E01', 'Baseline: one score for every ticket (the train rate)', 0.117, 0.000, 0.155, 'baseline', 'The floor: any useful model must be above it.'),
    ('E02', "Baseline: the rule 'flag priority 1', as a 0/1 score", 0.178, 0.008, 0.228, 'baseline', 'What Larkfield does today: the model must beat it.'),
    ('E03', 'Logistic regression, the 10 C04 features', 0.327, 0.012, 0.353, 'keep', 'Far above both baselines; train AP 0.328, so it does not overfit.'),
    ('E04', "E03 with class_weight='balanced'", 0.323, 0.012, None, 'reject', 'Lower than E03 on all 5 folds; one more setting for no gain.'),
    ('E05', "Baseline: the rule 'flag priority 1 or team payment'", 0.172, 0.006, None, 'reject', 'Lower than the simpler rule E02.'),
]
for row in EARLIER_ROWS:
    log_experiment(*row)
show_log()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Questions instead of weights

Logistic regression adds up the features, each with one weight. A **decision tree** asks yes-or-no questions about one feature at a time, such as "Is the priority 1?". Each answer sends a ticket left or right, to the next question, like a game of twenty questions. A **split** is one question and its two groups. A **leaf** is a group at the end. The **depth** is the number of questions from the top (the **root**) to a leaf. The tree's **score** for a ticket is the share of escalated train tickets in its leaf.

Run the next cell. It imports the tools and loads the train set.

In [ ]:
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text

from ticket_model import FEATURES, TARGET, build_pipeline, load

train = load("train")
print(len(train), "train tickets,", train[TARGET].sum(), "escalated")

> **Check.** You should see `16939 train tickets, 1983 escalated`.

Here are 10 made-up tickets, small enough to check by hand. Run the cell.

In [ ]:
tiny = pd.DataFrame({
    "ticket": [f"T-{i:02d}" for i in range(1, 11)],
    "priority": [1, 1, 1, 1, 2, 2, 2, 3, 3, 3],
    "word_count": [150, 40, 120, 45, 130, 160, 20, 30, 145, 60],
    "escalated": [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
})
print(tiny.to_string(index=False))

> **Check.** You should see 10 rows, T-01 to T-10. The escalated ones are T-01, T-03, T-04 and T-05.

## 2. Which question first?

A good question makes groups that are **pure**: all escalated, or all not escalated. **Impurity** measures how mixed a group is. The tree uses the Gini impurity: 2 × p × (1 − p), where p is the share of escalated tickets. A pure group has 0. A half-and-half group has 0.5.

By hand, for the 10 tickets: p = 0.4, so the impurity is 2 × 0.4 × 0.6 = **0.48**.

> **Predict.** Which question makes purer groups: "priority = 1?" or "more than 100 words?"? Write your guess, then run the cell.

In [ ]:
def impurity(escalated):
    """Gini impurity of a group: 2 × p × (1 − p), with p the share escalated."""
    p = escalated.mean()
    return 2 * p * (1 - p)


def impurity_after(question):
    """The impurity after a split: each group's impurity, weighted by its size."""
    yes, no = tiny[question], tiny[~question]
    return (len(yes) * impurity(yes["escalated"]) + len(no) * impurity(no["escalated"])) / len(tiny)


print("before any question: ", round(impurity(tiny["escalated"]), 3))
print("priority = 1?        ", round(impurity_after(tiny["priority"] == 1), 3))
print("more than 100 words? ", round(impurity_after(tiny["word_count"] > 100), 3))

> **Check.** You should see `0.48`, `0.317` and `0.4`.

"priority = 1?" lowers the impurity more. The tree tries every possible question and takes the one with the lowest impurity after the split. Then it asks again inside each group.

Run the next cell. It grows a tree of depth 2 on the 10 tickets, prints its questions, and scores a new ticket with priority 2 and 140 words.

In [ ]:
tiny_tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tiny_tree.fit(tiny[["priority", "word_count"]], tiny["escalated"])
print(export_text(tiny_tree, feature_names=["priority", "word_count"], show_weights=True))

new_ticket = pd.DataFrame({"priority": [2], "word_count": [140]})
print("score of the new ticket:", tiny_tree.predict_proba(new_ticket)[0, 1].round(3))

> **Check.** You should see `priority <= 1.50` first, then `word_count <= 42.50` on the priority 1 side and `word_count <= 95.00` on the other side. The score of the new ticket is `0.333`.

The second question uses a different cut point in each branch: 42.5 words for priority 1 and 95 words for the others. When the effect of one fact depends on another, the facts **interact**. A tree can learn that; logistic regression, with one weight for `word_count`, cannot. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/#which-question-first) draws this tree with the tickets in each leaf.

## 3. The real tree

Now the same on the 16,939 train tickets, with the 10 features. A tree compares a feature with a cut point. Scaling changes the numbers but not their order, so `tree_pipeline()` takes the scaler out (`"passthrough"` means "no step here"). Run the cell.

In [ ]:
def tree_pipeline(model):
    """build_pipeline() without the scaler: a tree does not need it."""
    return build_pipeline(model).set_params(prepare__numbers__scale="passthrough")


pipeline = tree_pipeline(DecisionTreeClassifier(max_depth=2, random_state=0))
pipeline.fit(train[FEATURES], train[TARGET])
names = list(pipeline.named_steps["prepare"].get_feature_names_out())
print(export_text(pipeline.named_steps["model"], feature_names=names, show_weights=True))

> **Check.** You should see `numbers__priority <= 1.50` first, then `numbers__word_count <= 66.50` and `numbers__word_count <= 100.50`. The four leaves have `[635.00, 128.00]`, `[1057.00, 565.00]`, `[7954.00, 507.00]` and `[5310.00, 783.00]`.

Priority 1 tickets with more than 66 words escalate most: 565 of 1,622, a score of 0.348. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/#the-real-tree) draws this tree.

## 4. Compare the tree with logistic regression

Use the rules of the loop: the same 5 folds, the same metric (average precision), and the result in the log. `evaluate()` returns the **train AP** (fit and scored on all train tickets) and the **CV AP** with its standard deviation.

> **Predict.** Which tree has the highest train AP? Which has the highest CV AP? Then run the cell. It takes a few seconds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def evaluate(pipeline):
    """Train AP (fit on all train tickets) and CV AP (mean ± std of the 5 folds)."""
    fold_scores = cross_val_score(pipeline, train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
    pipeline.fit(train[FEATURES], train[TARGET])
    train_ap = average_precision_score(train[TARGET], pipeline.predict_proba(train[FEATURES])[:, 1])
    return {"train AP": round(float(train_ap), 3), "CV AP": round(float(fold_scores.mean()), 3),
            "std": round(float(fold_scores.std()), 3)}


def table(results):
    """The results as a table, one row per model (rows still set to ... are left out)."""
    return pd.DataFrame({name: r for name, r in results.items() if isinstance(r, dict)}).T


results = {
    "logistic regression": evaluate(build_pipeline()),
    "tree, depth 3": evaluate(tree_pipeline(DecisionTreeClassifier(max_depth=3, random_state=0))),
    "tree, depth 6": evaluate(tree_pipeline(DecisionTreeClassifier(max_depth=6, random_state=0))),
    "tree, no limit": evaluate(tree_pipeline(DecisionTreeClassifier(random_state=0))),
}
print(table(results))

> **Check.** You should see these rows:
>
> | | train AP | CV AP | std |
> |---|---|---|---|
> | logistic regression | 0.328 | 0.327 | 0.012 |
> | tree, depth 3 | 0.244 | 0.234 | 0.011 |
> | tree, depth 6 | 0.346 | 0.258 | 0.010 |
> | tree, no limit | 1.000 | 0.137 | 0.004 |

The tree with no limit is perfect on train and the worst on tickets it did not see: it memorised the train tickets. The best tree, depth 6, is still far below logistic regression. The risk at Larkfield comes mostly from facts that each add some risk on their own, the shape of logistic regression.

## 5. Guided practice: log the trees

> **Your turn.** Write the rows E06 (depth 3), E07 (depth 6) and E08 (no limit) with `log_experiment()`. Take the CV AP and the std from `results`, for example `results["tree, depth 3"]["CV AP"]`. Use `None` for the valid AP, the decision `"reject"`, and a reason. Run the cell, then the check cell.

In [ ]:
# Your turn: write the rows E06, E07 and E08 (one log_experiment() call for each tree).
for row_id, name, change, reason in [
    ("E06", "tree, depth 3", "Decision tree, max_depth=3", "Far below E03."),
    ("E07", "tree, depth 6", "Decision tree, max_depth=6", "The best of the three trees, still far below E03."),
    ("E08", "tree, no limit", "Decision tree, no depth limit",
     "It memorises train (train AP 1.000) and fails on new tickets."),
]:
    log_experiment(row_id, change, results[name]["CV AP"], results[name]["std"], None, "reject", reason)
show_log()

In [ ]:
log = pd.read_csv(LOG_FILE).set_index("id")
missing = [row_id for row_id in ['E06', 'E07', 'E08'] if row_id not in log.index]
if missing:
    print("Not yet: the log is missing", ", ".join(missing) + ". Write the rows with log_experiment(), then run this cell again.")
else:
    rows = log.loc[['E06', 'E07', 'E08']]
    expect_hash("the log from E06 to E08", [[r.cv_ap, r.cv_std, None if pd.isna(r.valid_ap) else r.valid_ap, r.decision]
                                   for r in rows.itertuples()], "8794a467f02f605b")

> **Check.** You should see eight rows, and `Check passed: the log from E06 to E08 is right.`

## 6. Your turn: depth 4

Change one condition: `max_depth=4`.

> **Predict.** Will its CV AP be between depth 3 (0.234) and depth 6 (0.258)? Will its train AP be closer to its CV AP than for depth 6?

> **Your turn.** Replace `...` with `evaluate(tree_pipeline(DecisionTreeClassifier(...)))`. Run the cell, then the check cell.

In [ ]:
# Your turn: a tree with max_depth=4 (and random_state=0), scored with evaluate().
depth_4 = evaluate(tree_pipeline(DecisionTreeClassifier(max_depth=4, random_state=0)))
print(depth_4)

In [ ]:
if not isinstance(depth_4, dict):
    print("Not yet: replace ... with evaluate(tree_pipeline(DecisionTreeClassifier(...))).")
else:
    expect_hash("your result", depth_4, "2d4c9dbb243c0e24")

Compare your numbers with the answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/#your-turn-depth-4). Each extra level of questions fits the train tickets more closely, and the gap between train AP and CV AP grows. The next lesson follows this gap for every depth.

## 7. Failure case: a rule from two tickets

Tomás reads the depth-6 tree and finds a leaf with a score of 1.0: "priority 3 tickets with an order value between $190.53 and $190.80 always escalate". The next cell counts the small leaves of that tree, and shows the tickets in Tomás's leaf (leaf number 113). `apply()` gives the number of the leaf where each ticket ends. Run the cell.

In [ ]:
deep = tree_pipeline(DecisionTreeClassifier(max_depth=6, random_state=0)).fit(train[FEATURES], train[TARGET])
leaf_of_ticket = deep.named_steps["model"].apply(deep.named_steps["prepare"].transform(train[FEATURES]))
leaf_sizes = pd.Series(leaf_of_ticket).value_counts()
small = leaf_sizes[leaf_sizes < 10].index
pure = [leaf for leaf in small if train[TARGET][leaf_of_ticket == leaf].nunique() == 1]
print(len(leaf_sizes), "leaves;", len(small), "with fewer than 10 train tickets;", len(pure), "of these are pure")
print(train.loc[leaf_of_ticket == 113, ["ticket_id", "priority", "order_value", TARGET]].to_string(index=False))

> **Check.** You should see `59 leaves; 18 with fewer than 10 train tickets; 13 of these are pure`, then two priority 3 tickets with an order value near 190.6, both escalated.

The "rule" comes from two tickets. A leaf of two tickets gets a score of 0 or 1 very easily, by chance. **Never read a leaf's score without its number of tickets.** The next lesson limits the leaf size with `min_samples_leaf`. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/#a-rule-from-two-tickets) has the full failure case.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you grew a tree by impurity, read the real tree, compared three trees with logistic regression on the same folds, and logged them. No tree beats logistic regression (0.327 ± 0.012). A deep tree memorises its train tickets: the next lesson measures that and controls it.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Control complexity](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.